# Inheritance
- Allows creating new datatypes from existing ones
- Terminology: either parent class / child class or super class / sub class
- Child class "inherits" all attributes and methods of the parent class
    - Can choose to modify these as it wishes
- Recall that the `int` class has attributes such as `numerator`, `denominator`, `real`, `imag`
    - A bit unusual for integers to have these attributes -- a result of [\[PEP 3141 Link\]](https://peps.python.org/pep-3141/)
    - Forced a hierarchy among numeric datatypes `Number` $\Rightarrow$ `Complex` $\Rightarrow$ `Real` $\Rightarrow$ `Rational` $\Rightarrow$ `Integral`
- Inheritance is used to create more specialized datatypes from more generic ones
    - This subset hierarchy does exist in mathematics so saying that integer is a specific version of rationals makes sense
    - However, a side-effect of forcing that hierarchy onto Python is that `int` is forced to have needless attributes

# Mixed Fraction -- a case of unnecessary inheritance
- Not a good example of the use of inheritance
- Mixed fractions are just an alternate representation of rationals -- not a specialized subset
- The `__init__` method of the child class will usually call the `__init__` method of the parent class
    - **Warning**: creating a `MixedFraction` object will only call the `__init__` method of `MixedFraction` class
    - The `__init__` method of `Fraction` will not get called automatically
    - It is not necessary for a child `__init__` method to call the parent `__init__` method
    - If a child does not call the parent `__init__` method then it must do all the steps the parent was doing
        - For example, the `MixedFraction' class inherits the `num` and `den` attributes from its parent
        - If it does not want to call the parent `__init__` method, then it must initialize `num` and `den` itself
        - It must do all the typechecks itself too -- in most cases, it is convenient to just call the parent `__init__` method

In [1]:
class Fraction:
    # Class attributes
    prec = 2      # How many digits of precision to use when printing a Fraction in decimal form?
    pop = 0       # How many Fraction objects exist in the universe?
    
    def __init__( self, num = 0, den = 1 ):
        # guard clauses to check for type integrity
        if not type( num ) == int:
            raise TypeError( "numerator must be an integer" )
        if not type( den ) == int:
            raise TypeError( "denominator must be an integer" )
        if den == 0:
            raise ValueError( "denominator cannot be zero" )
        self.n = num
        self.d = den
        self.__p = f"my original value was {self}"
        # This will ensure that creation of a MixedFraction or Percentage does not affect Fraction population
        if type( self ) == Fraction:
            Fraction.pop += 1

    def __frac__( s ):
        # Defining new object attributes allowed outside __init__ but not advisable
        s.new_attr = 42
        return f"{s.n / s.d:.{Fraction.prec}f}"

    # Return a printable representation of the object **value**
    def __str__( myself ):
        return f"{myself.n} / {myself.d}"

    # Return a description of how this object can be **created**
    # Mostly used in debugging
    def __repr__( s ):
        return f"Fraction( {s.n}, {s.d} )"

    def __add__( s, o ):
        if not type( o ) == Fraction:
            if type( o ) == int:
                o = Fraction( o )
            else:
                raise TypeError( "can only add Fraction to Fraction or int" )
        return Fraction( s.n * o.d + s.d * o.n, s.d * o.d )

    # Be careful -- if we return o + s, it will cause infinite recursion
    def __radd__( s, o ):
        return s + o

    def __sub__( myself, thyself ):
        return Fraction( myself.n * thyself.d - myself.d * thyself.n, myself.d * thyself.d )

    # Be careful -- if we return o - s, it will cause infinite recursion
    def __rsub__( s, o ):
        return -s + o

    def __neg__( s ):
        return Fraction( -s.n, s.d )

    def __iadd__( s, o ):
        return s + o
        
    # Use Euclid's algo to compute the GCD
    # Note that this method does not create a new object
    # Instead it modifies the object itself
    def reduce( s ):
        x, y = ( s.n, s.d )
        while y != 0:
            x, y = y, x % y
        s.n //= abs( x )
        s.d //= abs( x )
        # Class methods can access the "unmangled" names
        print( s.__p )

    # This will ensure that deletion of a MixedFraction or Percentage does not affect Fraction population
    def __del__( s ):
        if type( s ) == Fraction:
            Fraction.pop -= 1

In [2]:
class MixedFraction( Fraction ):
    def __init__( s, num = 0, den = 1 ):
        s.new_attr = 42 # child can define new attributes in addition to those inherited from parent
        super().__init__( num, den ) # not necessary but convenient in most situations
    def __str__( s ):
        if abs( s.n ) < abs( s.d ):
            return f"{s.n} / {s.d}"
        else:
            return f" {s.n // s.d} ( {s.n % s.d} / {s.d} ) "

In [3]:
mx1 = MixedFraction( 10, 3 )
print( mx1 )

 3 ( 1 / 3 ) 


# Percentage -- a better example of inheritance
- Percentages are a special type of non-negative fractions that lie between $0$ and $1$
- This makes them a better candidate for a sub class / child class of `Fraction`

In [4]:
class Percentage( Fraction ):
    def __init__( s, num = 0, den = 1 ):
        if num < 0:
            raise ValueError( "numerator cannot be negative" )
        if den < 0:
            raise ValueError( "denominator cannot be negative" )
        if num > den:
            raise ValueError( "not a percentage" )
        super().__init__( num, den )
    def __str__( s ):
        return f"{s.n / s.d * 100:.{Fraction.prec}f}%"

In [5]:
# Due to the way we defined the __init__ and __del__ methods for Fraction,
# creation of child class objects will not affect the class attribute of the parent class
print( Fraction.pop )
p = Percentage( 2, 3 )
# Fraction()
print( p )
print( Fraction.pop )
f = Fraction( 42, 2 )
print( Fraction.pop )

0
66.67%
0
1
